# Myllia: Direction Notebook (Bilinear Conditional Factorization)

This notebook implements a medium-large architecture shift:

- Learn a low-rank interaction between perturbed gene embeddings and output gene embeddings
- Predict the full delta vector as a bilinear form with rank `R`
- Train with a metric-aligned weighted L1 proxy and evaluate with the official `myllia_score`

Core model:
\[
\hat D_{i,j} = \langle W_p z_{g_i},\; W_o u_j \rangle + b_j + b_i
\]
where:
- `z_{g_i}` is an embedding for the perturbed gene `g_i`
- `u_j` is an embedding for output gene `j`
- `R` is a small rank (16 to 64)

This uses `training_cells.h5ad` to build output gene embeddings.


In [144]:
import numpy as np
import pandas as pd
from pathlib import Path

import anndata as ad
import scanpy as sc
from scipy import sparse

from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold

import torch
import torch.nn as nn

SEED = 6
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(DEVICE)

EMB_DIM_PERT = 128   # pert gene embedding dim (from SVD)
EMB_DIM_OUT = 128   # output gene embedding dim (from SVD)
RANK_R = 48    # low-rank interaction size

DROPOUT = 0.10
LR = 2e-3 * 0.6 * 0.2
WD = 1e-4 * 2
EPOCHS = 400
BATCH_GENES = 32 # minibatch over perturbed genes
EVAL_EVERY = 5
PATIENCE = 10 # early stopping patience in eval steps

# Gate parameters should match metric structure
GATE_A = 0.0
GATE_B = 0.2
EPS = 1e-12

ROOT = Path(".")

# --- added: CV alpha sweep + refit ---
ALPHA_GRID = np.linspace(0.7, 1.0, 30).astype(np.float32).tolist()
MODEL_SEEDS = [90, 70, 80]
GRAD_CLIP = 1.0
H5AD_PATH = ROOT / "data" / "training_cells.h5ad"  # used ONLY for perts not in gene_columns

In [145]:
means_path = ROOT / "data" / "training_data_means.csv"
valmap_path = ROOT / "data" / "pert_ids_val.csv"
sample_sub_path = ROOT / "data" / "sample_submission.csv"

df_means = pd.read_csv(means_path)
df_valmap = pd.read_csv(valmap_path)
df_sub = pd.read_csv(sample_sub_path)

gene_columns = [c for c in df_means.columns if c != "pert_symbol"]

baseline_mask = df_means["pert_symbol"].astype(str) == "non-targeting"
x_base = df_means.loc[baseline_mask, gene_columns].iloc[0].to_numpy(np.float32)

df_train = df_means.loc[~baseline_mask].reset_index(drop=True)
train_genes = df_train["pert_symbol"].astype(str).to_numpy()

X_train_means = df_train[gene_columns].to_numpy(np.float32)
D_train = X_train_means - x_base[None, :] # (80, 5127) delta vs non-targeting

delta_baseline = D_train.mean(axis=0).astype(np.float32)

# pert_id -> gene symbol for leaderboard (first 60)
val_map = dict(zip(df_valmap["pert_id"].astype(str), df_valmap["pert"].astype(str)))

print("Train perts:", len(train_genes), "G:", len(gene_columns))
print("Sample submission rows:", len(df_sub))
print("Val mapping entries:", len(val_map))

Train perts: 80 G: 5127
Sample submission rows: 120
Val mapping entries: 60


In [174]:
# h5ad augmentation (used during training, CV-safe mapping must be fit per fold)
AUGMENT_H5AD = True     # turn on/off augmentation
AUG_P = 1               # 1.0 = full bootstrap targets, 0.5 = half bootstrap, 0 = off
BOOT_K = 32             # number of bootstrap samples per perturbation
BOOT_M = 256            # cells per bootstrap sample
BOOT_SEED = SEED        # reproducible precompute

# IMPORTANT: to avoid CV leakage, fit h5ad->means affine mapping PER FOLD using tr_idx only.
AUG_SLOPE_CLAMP_MIN = 0.2
AUG_SLOPE_CLAMP_MAX = 5.0
AUG_MIN_FIT_PERTS = 8   # if fewer training perts have h5ad cells, skip augmentation in that fold

# Always fill missing perts from h5ad (DO NOT TURN OFF)
FILL_MISSING_PERTS = True
H5AD_TOPK = 256

# Turn OFF all external/ctrl feature experiments (LB liked none)
USE_UOUT_CTRL = False
USE_ZCTRL_ALL = False

# Explicitly keep external sources off
USE_GENEPT = False
USE_DOROTHEA = False
USE_STRING_SEQ = False
USE_STRING_NET = False

# Path to training h5ad
H5AD_PATH = ROOT / "data" / "training_cells.h5ad"

BASIS_K = 96
HNET_HIDDEN = 192
HNET_DROPOUT = 0.10

LAMBDA_BASIS_DRIFT = 5e-5
LAMBDA_BASIS_GAIN  = 1e-4

EXP_NAME = "basis_hypernet_k96_h192"

In [147]:
# ============================================================
# MINIMAL H5AD HELPERS (FOR FILL + AUG)
# - load h5ad
# - find perturbation column
# - normalize CPM10K + log2(1+x)
# - build control cache
# - build missing-pert Z via ctrl correlation
# ============================================================

import anndata as ad
import scipy.sparse as sp

_H5AD_CACHE = None

def _pick_pert_col(adata):
    # Try common obs column names across variants
    for c in ["sgrna_symbol", "pert_symbol", "pert", "perturbation", "gene", "target_gene", "target"]:
        if c in adata.obs.columns:
            return c
    raise ValueError("Could not find perturbation column in h5ad obs.")

def load_h5ad_ctrl_cache(h5ad_path: Path, gene_columns):
    """
    Cache:
      - Xn (CPM10K + log2(1+x)) sparse
      - ctrl mask
      - var map (GENE->idx)
      - out_idx mapping for the output gene columns
      - Xout_z (control cells, standardized per output gene)
      - obs_pertU
    """
    global _H5AD_CACHE
    if _H5AD_CACHE is not None:
        return _H5AD_CACHE

    adata = ad.read_h5ad(str(h5ad_path))
    pert_col = _pick_pert_col(adata)

    Xc = adata.X
    if not sp.issparse(Xc):
        Xc = sp.csr_matrix(Xc)
    else:
        Xc = Xc.tocsr()

    # CPM10K normalize, then log2(1+x)
    cell_sum = np.asarray(Xc.sum(axis=1)).ravel().astype(np.float64)
    scale = (10000.0 / np.clip(cell_sum, 1e-12, None)).astype(np.float64)
    Xn = Xc.multiply(scale[:, None]).tocsr()
    Xn.data = np.log1p(Xn.data) / np.log(2.0)

    obs_pert = adata.obs[pert_col].astype(str).to_numpy()
    obs_pertU = np.char.upper(obs_pert.astype("U"))

    # Control detection (keep simple; tune if your file uses different token)
    ctrl_mask = (obs_pertU == "NON-TARGETING")
    if int(ctrl_mask.sum()) == 0:
        raise ValueError("No NON-TARGETING control cells found in h5ad. If your controls use a different label, update ctrl_mask.")

    var_names = adata.var_names.astype(str).to_numpy()
    varU = np.char.upper(var_names.astype("U"))
    var = {varU[i]: i for i in range(len(varU))}

    out_idx = np.array([var[str(g).upper()] for g in gene_columns], dtype=np.int64)

    Xn_ctrl = Xn[ctrl_mask]
    Xout = Xn_ctrl[:, out_idx]
    if sp.issparse(Xout):
        Xout = Xout.toarray()
    Xout = Xout.astype(np.float32)

    # z-score per output gene over control cells
    mu = Xout.mean(axis=0, keepdims=True)
    sd = Xout.std(axis=0, keepdims=True) + 1e-6
    Xout_z = ((Xout - mu) / sd).astype(np.float32)

    _H5AD_CACHE = dict(
        Xn=Xn,
        Xn_ctrl=Xn_ctrl,
        ctrl_mask=ctrl_mask,
        var=var,
        out_idx=out_idx,
        Xout_z=Xout_z,
        obs_pertU=obs_pertU,
    )
    print("[h5ad] cache built:",
          "n_cells=", Xn.shape[0],
          "n_genes=", Xn.shape[1],
          "n_ctrl=", int(ctrl_mask.sum()))
    return _H5AD_CACHE

def build_z_from_ctrl_corr(cache, genesU, P_out, topk=256):
    """
    Build a Z embedding for each gene in genesU using correlation of that gene's
    control-cell expression with standardized output genes (Xout_z), projected
    into the SVD basis P_out (G x d_pert).

    Returns dict: {geneU: z (d_pert,)}
    """
    Xout_z = cache["Xout_z"]      # (n_ctrl, G_out) standardized
    Xn_ctrl = cache["Xn_ctrl"]    # (n_ctrl, G_all) CPM/log space
    var = cache["var"]

    out = {}
    for gU in genesU:
        j = var.get(str(gU).upper(), None)
        if j is None:
            continue

        xg = Xn_ctrl[:, j]
        if sp.issparse(xg):
            xg = xg.toarray()
        xg = np.asarray(xg).ravel().astype(np.float32)
        xg = (xg - xg.mean()) / (xg.std() + 1e-6)

        corr = (xg[:, None] * Xout_z).mean(axis=0)  # (G_out,)
        idx = np.argsort(-np.abs(corr))[:int(topk)]
        w = corr[idx].astype(np.float32)

        z = (w[:, None] * P_out[idx]).sum(axis=0)   # (d_pert,)
        z = z / (np.linalg.norm(z) + 1e-12)
        out[str(gU).upper()] = z.astype(np.float32)

    return out

Build gene embeddings from D_train (SVD on signed-log transformed deltas). Build embeddings for output genes (gene_columns) via SVD on (80, 5127). Pert embeddings are looked up by gene symbol in gene_columns, otherwise fallback to the mean embedding.

In [148]:
val_targets = df_valmap["pert"].astype(str).tolist()

genes_needed = sorted(set([str(g).upper() for g in train_genes.tolist()] +
                          [str(g).upper() for g in val_targets]))

geneU = pd.Index([str(g).upper() for g in gene_columns])

if "SVD_SIGMA_POWER" not in globals():
    SVD_SIGMA_POWER = 0.5
if "SVD_ROW_NORM" not in globals():
    SVD_ROW_NORM = True

def _row_l2_normalize(x, eps=1e-12):
    x = np.asarray(x, np.float32)
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return (x / np.clip(n, eps, None)).astype(np.float32)

def _vec_l2_normalize(x, eps=1e-12):
    x = np.asarray(x, np.float32)
    n = np.linalg.norm(x)
    return (x / max(float(n), eps)).astype(np.float32)

# signed log transform (handles negative deltas)
X = D_train.astype(np.float32, copy=True)
X = np.sign(X) * np.log2(1.0 + np.abs(X))

k_req = max(int(EMB_DIM_PERT), int(EMB_DIM_OUT))
k_max = min(X.shape[0] - 1, X.shape[1] - 1)
k_svd = min(k_req, k_max)

svd = TruncatedSVD(n_components=k_svd, random_state=SEED)
svd.fit(X)

V = svd.components_.T.astype(np.float32)                 # (G, k)
S = svd.singular_values_.astype(np.float32)              # (k,)

d_pert = min(int(EMB_DIM_PERT), int(k_svd))
d_out  = min(int(EMB_DIM_OUT),  int(k_svd))

# ------------------------------------------------------------
# ONE CHANGE:
# scale latent directions by singular value**power
# power=0.0 -> old behavior (pure V)
# power=1.0 -> full VS
# power=0.5 -> compromise
# ------------------------------------------------------------
scale_pert = np.power(np.clip(S[:d_pert], 1e-8, None), float(SVD_SIGMA_POWER)).astype(np.float32)
scale_out  = np.power(np.clip(S[:d_out ], 1e-8, None), float(SVD_SIGMA_POWER)).astype(np.float32)

gene_emb_pert_all = (V[:, :d_pert] * scale_pert[None, :]).astype(np.float32)   # (G, d_pert)
gene_emb_out_all  = (V[:, :d_out ] * scale_out [None, :]).astype(np.float32)    # (G, d_out)

if SVD_ROW_NORM:
    gene_emb_pert_all = _row_l2_normalize(gene_emb_pert_all)
    gene_emb_out_all  = _row_l2_normalize(gene_emb_out_all)

print("SVD k:", k_svd)
print("Effective dims:", "d_pert=", d_pert, "d_out=", d_out)
print("SVD_SIGMA_POWER:", float(SVD_SIGMA_POWER), "SVD_ROW_NORM:", bool(SVD_ROW_NORM))
print("top singular values:", np.round(S[:10], 4).tolist())

# base dicts (only genes in gene_columns exist here)
gene2emb_pert_svd = {
    str(gene_columns[i]).upper(): gene_emb_pert_all[i].copy()
    for i in range(len(gene_columns))
}
gene2emb_out_svd = {
    str(gene_columns[i]).upper(): gene_emb_out_all[i].copy()
    for i in range(len(gene_columns))
}

emb_fallback_pert = _vec_l2_normalize(gene_emb_pert_all.mean(axis=0))
emb_fallback_out  = _vec_l2_normalize(gene_emb_out_all.mean(axis=0))

# Missing perts = train/val perturbation genes not in gene_columns
missing_train = [g for g in train_genes.tolist() if str(g).upper() not in geneU]
missing_val   = [g for g in val_targets           if str(g).upper() not in geneU]
missing_allU  = sorted(set([str(g).upper() for g in (missing_train + missing_val)]))

if missing_train:
    print(f"train perts not in gene_columns: {len(missing_train)}. Example: {missing_train[:12]}")
if missing_val:
    print(f"val perts not in gene_columns: {len(missing_val)}. Example: {missing_val[:12]}")

# h5ad: fill embeddings for missing perts only
missing_pert_h5ad = {}

if FILL_MISSING_PERTS and (len(missing_allU) > 0):
    cache = load_h5ad_ctrl_cache(H5AD_PATH, gene_columns)

    # use perturbation-side embedding space for missing perturbation fills
    P_out = gene_emb_pert_all.astype(np.float32)  # (G, d_pert)

    tmp = build_z_from_ctrl_corr(cache, missing_allU, P_out, topk=int(H5AD_TOPK))

    for gU, z in tmp.items():
        z = np.asarray(z, np.float32)
        if SVD_ROW_NORM:
            z = _vec_l2_normalize(z)
        missing_pert_h5ad[gU] = z.astype(np.float32)

    print("[h5ad] embedded missing perts:", len(missing_pert_h5ad), "of", len(missing_allU))

# U_out is pure SVD-derived geometry
U_out = np.vstack([
    gene2emb_out_svd.get(str(g).upper(), emb_fallback_out)
    for g in gene_columns
]).astype(np.float32)   # (G, d_out)

# emb_pert: SVD if possible, else h5ad fill if missing, else fallback
def emb_pert(g: str) -> np.ndarray:
    gU = str(g).upper()
    if gU in gene2emb_pert_svd:
        return gene2emb_pert_svd[gU].astype(np.float32)
    if gU in missing_pert_h5ad:
        return missing_pert_h5ad[gU].astype(np.float32)
    return emb_fallback_pert.astype(np.float32)

# Pert embeddings for the training perts (N ~ 80)
Z_train = np.vstack([emb_pert(g) for g in train_genes]).astype(np.float32)

print("EXP_NAME:", EXP_NAME)
print("U_out:", U_out.shape, "Z_train:", Z_train.shape)
print("U_out row-norm mean:", float(np.linalg.norm(U_out, axis=1).mean()))
print("Z_train row-norm mean:", float(np.linalg.norm(Z_train, axis=1).mean()))

SVD k: 79
Effective dims: d_pert= 79 d_out= 79
SVD_SIGMA_POWER: 0.5 SVD_ROW_NORM: True
top singular values: [18.514999389648438, 14.621999740600586, 12.474800109863281, 10.648900032043457, 9.255999565124512, 8.270400047302246, 8.001199722290039, 7.068999767303467, 6.949699878692627, 6.755099773406982]
train perts not in gene_columns: 8. Example: ['BRD4', 'CHD4', 'DNAJA3', 'INO80', 'KAT8', 'KDM4A', 'PMEL', 'SETD1A']
val perts not in gene_columns: 8. Example: ['SMARCB1', 'PSMA1', 'CUL1', 'FLT4', 'FOXH1', 'HK2', 'TRAM2', 'DPH2']
[h5ad] cache built: n_cells= 17882 n_genes= 19226 n_ctrl= 1026
[h5ad] embedded missing perts: 16 of 16
EXP_NAME: basis_hypernet_score_aligned_loss
U_out: (5127, 79) Z_train: (80, 79)
U_out row-norm mean: 0.9996098875999451
Z_train row-norm mean: 1.0


In [149]:
def gate_smoothstep(x, a=GATE_A, b=GATE_B):
    t = (x - a) / (b - a)
    t = torch.clamp(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)

def per_row_weighted_l1_like(
    delta_true: torch.Tensor,   # (N, G)
    delta_pred: torch.Tensor,   # (N, G)
    eps: float = EPS
) -> torch.Tensor:
    """
    Per-row weighted L1-like using w**1.5 on gate(|delta_true|).
    Returns: (N,) per-row losses.
    """
    w = gate_smoothstep(torch.abs(delta_true), a=GATE_A, b=GATE_B)  # (N, G)
    w2 = w ** 1.5

    err = torch.abs(delta_pred - delta_true)                       # (N, G)
    num = torch.sum(w2 * err, dim=1)                               # (N,)
    den = torch.clamp(torch.sum(w2, dim=1), min=eps)               # (N,)
    return num / den                                               # (N,)

def weighted_l1_like_rowweighted(
    delta_true: torch.Tensor,     # (N, G)
    delta_pred: torch.Tensor,     # (N, G)
    baseline_wmae: torch.Tensor,  # (N,)
    *,
    eps: float = 1e-8,
    mode: str = "inv_sqrt",       # "inv", "inv_sqrt", "inv_log"
    clamp_min: float = 0.5,
    clamp_max: float = 3.0,
) -> torch.Tensor:
    """
    Row-weighted version of per_row_weighted_l1_like.

    Weight idea:
      - baseline_wmae small => ratio metric is unforgiving => upweight that row
      - baseline_wmae large => easier => downweight a bit

    Returns: scalar loss
    """
    per_row = per_row_weighted_l1_like(delta_true, delta_pred, eps=eps)  # (N,)

    b = baseline_wmae.to(delta_true.device).to(delta_true.dtype)

    if mode == "inv":
        w = 1.0 / (b + eps)
    elif mode == "inv_sqrt":
        w = 1.0 / torch.sqrt(b + eps)
    elif mode == "inv_log":
        w = 1.0 / torch.log1p(b + eps)
    else:
        raise ValueError(f"Unknown mode={mode}")

    # clamp to prevent a few rows from dominating training
    w = torch.clamp(w, min=clamp_min, max=clamp_max)

    # normalized weighted mean (stable)
    return torch.sum(w * per_row) / torch.clamp(torch.sum(w), min=eps)

In [150]:
import torch
import torch.nn as nn

class LowRankBasisHyperNet(nn.Module):
    def __init__(self, d_pert, basis_init, hidden, dropout):
        super().__init__()

        G, K = basis_init.shape
        self.G = G
        self.K = K

        self.net = nn.Sequential(
            nn.Linear(d_pert, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, K),
        )

        # fixed starting basis
        self.register_buffer("basis0", basis_init.detach().clone())   # (G, K)

        # learn a small drift away from the starting basis
        self.basis_delta = nn.Parameter(torch.zeros_like(self.basis0))  # (G, K)

        # per-component gain
        self.basis_gain = nn.Parameter(torch.ones(K, device=basis_init.device))  # (K,)

        # output biases
        self.bias_gene = nn.Parameter(torch.zeros(G, device=basis_init.device))   # (G,)
        self.bias_global = nn.Parameter(torch.zeros(1, device=basis_init.device)) # (1,)

    def set_gene_bias_init(self, bias_init):
        with torch.no_grad():
            self.bias_gene.copy_(bias_init)

    def encode(self, z_pert):
        return self.net(z_pert)   # (B, K)

    def get_basis(self):
        return self.basis0 * self.basis_gain[None, :] + self.basis_delta   # (G, K)

    def forward(self, z_pert):
        c = self.encode(z_pert)      # (B, K)
        B = self.get_basis()         # (G, K)
        y = c @ B.T                  # (B, G)
        y = y + self.bias_gene[None, :] + self.bias_global
        return y

In [151]:
# ============================================
# BASIS BUILD + TENSORS
# - keep your normal embedding cell for Z_train / emb_pert
# - this cell builds a GLOBAL response basis from Y
# ============================================

import numpy as np
import torch
from sklearn.decomposition import TruncatedSVD

Y = D_train.astype(np.float32)
G = Y.shape[1]
N = Y.shape[0]

basis_k_eff = min(int(BASIS_K), int(N - 1), int(G - 1))
if basis_k_eff < 2:
    raise ValueError(f"Invalid BASIS_K after clipping: {basis_k_eff}")

# Build basis in target space
# center across perturbations so basis captures variation, not just mean offset
Y_center = Y - Y.mean(axis=0, keepdims=True)

svd_basis = TruncatedSVD(n_components=basis_k_eff, random_state=SEED)
svd_basis.fit(Y_center)

# (G, K) orthonormal-ish directions in target space
BASIS_INIT = svd_basis.components_.T.astype(np.float32)

# column normalize for stable coefficient learning
BASIS_INIT = BASIS_INIT / np.clip(
    np.linalg.norm(BASIS_INIT, axis=0, keepdims=True), 1e-12, None
).astype(np.float32)

# initialize per-gene bias to mean target over training perturbations
GENE_BIAS_INIT = Y.mean(axis=0).astype(np.float32)

Zt = torch.tensor(Z_train, device=device, dtype=torch.float32)              # (N, d_pert)
Yt = torch.tensor(Y, device=device, dtype=torch.float32)                    # (N, G)
BASIS_INIT_t = torch.tensor(BASIS_INIT, device=device, dtype=torch.float32) # (G, K)
GENE_BIAS_INIT_t = torch.tensor(GENE_BIAS_INIT, device=device, dtype=torch.float32)

print("EXP_NAME:", EXP_NAME)
print("N:", N, "G:", G, "device:", device)
print("Z_train:", Z_train.shape)
print("BASIS_INIT:", BASIS_INIT.shape)
print("basis explained variance ratio sum:", float(svd_basis.explained_variance_ratio_.sum()))

EXP_NAME: basis_hypernet_score_aligned_loss
N: 80 G: 5127 device: cuda
Z_train: (80, 79)
BASIS_INIT: (5127, 79)
basis explained variance ratio sum: 1.0


In [152]:
gt_df = pd.read_csv("Data/training_data_ground_truth_table.csv")
sol_aligned = gt_df.set_index("pert_id").loc[train_genes].reset_index()

# Keep only needed columns for speed: pert_id + genes + weights + baseline
w_cols = [f"w_{g}" for g in gene_columns]
sol_aligned = sol_aligned[["pert_id"] + list(gene_columns) + w_cols + ["baseline_wmae"]]
baseline_wmae = sol_aligned["baseline_wmae"].to_numpy(np.float32)
baseline_wmae_t = torch.tensor(baseline_wmae, device=device, dtype=torch.float32)

W_gene = sol_aligned[w_cols].to_numpy(np.float32)   # (N, G)

W64 = W_gene.astype(np.float64)
row_sums = W64.sum(axis=1, keepdims=True)
W64 *= (G / np.maximum(row_sums, 1e-300))
W64[:, -1] += (G - W64.sum(axis=1))
W_gene = W64.astype(np.float32)

In [153]:
# Score provided by the Competition
def score_delta(dt, dp, idx):
    dt = np.asarray(dt, np.float64)
    dp = np.asarray(dp, np.float64)
    w  = W_gene[idx].astype(np.float64, copy=False)
    base = baseline_wmae[idx].astype(np.float64, copy=False)

    abs_err = np.abs(dt - dp)
    pred_wmae = np.mean(abs_err * w, axis=1)
    pred_wmae = np.maximum(pred_wmae, 1e-12)
    base = np.maximum(base, 1e-12)

    terms = np.log2(base / pred_wmae)
    terms = np.minimum(terms, 5.0)
    sum_wmae = float(np.sum(terms))
    mean_term = float(np.mean(terms))

    a = dp.ravel()
    b = dt.ravel()
    x = np.maximum(np.abs(a), np.abs(b))
    t = np.clip(x / 0.2, 0.0, 1.0)
    w_gate = t * t * (3.0 - 2.0 * t)
    w2 = w_gate * w_gate

    num = np.sum(w2 * a * b)
    den = np.sqrt(np.sum(w2 * a * a)) * np.sqrt(np.sum(w2 * b * b))
    wcos = 0.0 if den < 1e-12 else float(num / den)
    wcos_pos = max(0.0, wcos)

    raw = sum_wmae * wcos_pos
    score = round(raw, 5)

    # normalized (scale-free) metric
    score_per_row = mean_term * wcos_pos

    return {
        "score": score,
        "raw": float(raw),
        "sum_wmae": sum_wmae,
        "mean_term": mean_term,
        "wcos": wcos,
        "score_per_row": float(score_per_row),
        "n_rows": int(len(idx)),
    }

In [154]:
# EXTRA H5AD DATA: BOOTSTRAP DELTAS
boot_raw = None
h5_mean_raw = None
boot_ok = None

if AUGMENT_H5AD:
    cache = load_h5ad_ctrl_cache(H5AD_PATH, gene_columns)

    Xn = cache["Xn"]               # sparse (n_cells, 19226) normalized CPM10K + log2
    out_idx = cache["out_idx"]     # (5127,)
    obs_pertU = cache["obs_pertU"] # (n_cells,) uppercase pert labels
    ctrl_mask = cache["ctrl_mask"]

    # control mean over output genes (in same h5ad normalized space)
    ctrl_mean = Xn[ctrl_mask][:, out_idx].mean(axis=0)
    if sparse.issparse(ctrl_mean):
        ctrl_mean = ctrl_mean.A
    ctrl_mean = np.asarray(ctrl_mean).ravel().astype(np.float32)  # (5127,)

    N = len(train_genes)
    G = len(gene_columns)

    h5_mean_raw = np.zeros((N, G), dtype=np.float32)
    boot_raw = np.zeros((N, BOOT_K, G), dtype=np.float32)
    boot_ok = np.zeros((N,), dtype=np.int32)

    rng = np.random.RandomState(BOOT_SEED)

    for i, g in enumerate(train_genes.tolist()):
        gU = str(g).upper()
        rows = np.where(obs_pertU == gU)[0]
        if len(rows) == 0:
            continue

        boot_ok[i] = 1

        # mean delta using ALL cells for this pert
        Xi = Xn[rows][:, out_idx].mean(axis=0)
        if sparse.issparse(Xi):
            Xi = Xi.A
        Xi = np.asarray(Xi).ravel().astype(np.float32)
        h5_mean_raw[i] = (Xi - ctrl_mean)

        # bootstraps
        for k in range(BOOT_K):
            samp = rng.choice(rows, size=min(int(BOOT_M), len(rows)), replace=True)
            Xk = Xn[samp][:, out_idx].mean(axis=0)
            if sparse.issparse(Xk):
                Xk = Xk.A
            Xk = np.asarray(Xk).ravel().astype(np.float32)
            boot_raw[i, k] = (Xk - ctrl_mean)

    print("[aug] boot_ok:", int(boot_ok.sum()), "/", N)
    print("[aug] h5_mean_raw:", h5_mean_raw.shape, "boot_raw:", boot_raw.shape)

[aug] boot_ok: 80 / 80
[aug] h5_mean_raw: (80, 5127) boot_raw: (80, 32, 5127)


In [ ]:
# ================================
# CV TRAINING CELL
# - low-rank response-basis hypernetwork
# - GROUPED CV BY PERTURBATION IDENTITY
# - validation perts are fully unseen during training
# - prints ONE summary line per fold (means across seeds)
# - keeps OOF preds + global alpha sweep
# ================================
PERT_NAMES = np.asarray(train_genes)
import numpy as np
import torch
from sklearn.model_selection import GroupKFold

ALPHA_GRID = np.linspace(0.3, 1.1, 100).astype(np.float32).tolist()

def apply_shrink(pred, baseline, alpha):
    return float(alpha) * pred + (1.0 - float(alpha)) * baseline[None, :]

def train_one_fold(tr_idx, va_idx, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = LowRankBasisHyperNet(
        d_pert=Zt.shape[1],
        basis_init=BASIS_INIT_t,
        hidden=HNET_HIDDEN,
        dropout=HNET_DROPOUT
    ).to(device)

    model.set_gene_bias_init(GENE_BIAS_INIT_t)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    tr_idx = np.asarray(tr_idx)
    va_idx = np.asarray(va_idx)
    va_idx_t = torch.tensor(va_idx, device=device, dtype=torch.long)

    aug_enabled = (
        AUGMENT_H5AD
        and (boot_raw is not None)
        and (h5_mean_raw is not None)
        and (boot_ok is not None)
    )

    slopes_t = None
    intercepts_t = None
    rng_aug = None

    if aug_enabled:
        ok_mask = (boot_ok[tr_idx] == 1)
        tr_fit = tr_idx[ok_mask]

        if len(tr_fit) >= AUG_MIN_FIT_PERTS:
            A = h5_mean_raw[tr_fit].astype(np.float32)  # (n_fit, G)
            B = Y[tr_fit].astype(np.float32)            # (n_fit, G)

            Am = A.mean(axis=0)
            Bm = B.mean(axis=0)
            Av = ((A - Am[None, :]) ** 2).mean(axis=0) + 1e-6
            Cov = ((A - Am[None, :]) * (B - Bm[None, :])).mean(axis=0)

            slopes = (Cov / Av).astype(np.float32)
            slopes = np.clip(
                slopes,
                float(AUG_SLOPE_CLAMP_MIN),
                float(AUG_SLOPE_CLAMP_MAX)
            ).astype(np.float32)
            intercepts = (Bm - slopes * Am).astype(np.float32)

            slopes_t = torch.tensor(slopes[None, :], device=device, dtype=torch.float32)
            intercepts_t = torch.tensor(intercepts[None, :], device=device, dtype=torch.float32)

            rng_aug = np.random.RandomState(seed + 2027)
        else:
            aug_enabled = False

    best_score = -1e18
    best_alpha = 0.0
    best_epoch = 0
    best_state = None
    best_va_pred = None
    best_mean_term = 0.0
    best_wcos = 0.0
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = tr_idx.copy()
        np.random.shuffle(perm)

        for start in range(0, len(perm), BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t))

            if aug_enabled and (slopes_t is not None):
                kk = rng_aug.randint(0, BOOT_K, size=len(b))
                dt_raw_np = boot_raw[b, kk, :]
                dt_raw_t = torch.tensor(dt_raw_np, device=device, dtype=torch.float32)

                dt_aug = dt_raw_t * slopes_t + intercepts_t
                dt_b = (1.0 - float(AUG_P)) * Yt.index_select(0, b_t) + float(AUG_P) * dt_aug
            else:
                dt_b = Yt.index_select(0, b_t)

            bw_b = baseline_wmae_t.index_select(0, b_t)

            loss_main = weighted_l1_like_rowweighted(
                dt_b, pred, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )

            loss_basis_drift = LAMBDA_BASIS_DRIFT * torch.mean(model.basis_delta * model.basis_delta)
            loss_basis_gain  = LAMBDA_BASIS_GAIN  * torch.mean((model.basis_gain - 1.0) ** 2)

            loss = loss_main + loss_basis_drift + loss_basis_gain

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                va_pred = model(Zt.index_select(0, va_idx_t)).detach().cpu().numpy().astype(np.float32)

            va_true = Y[va_idx]

            sc_best = -1e18
            a_best = 0.0
            best_dbg = None

            for a in ALPHA_GRID:
                pred_a = apply_shrink(va_pred, delta_baseline, float(a))
                out = score_delta(va_true, pred_a, va_idx)
                sc = out["score"]
                if sc > sc_best:
                    sc_best = sc
                    a_best = float(a)
                    best_dbg = out

            if sc_best > best_score:
                best_score = float(sc_best)
                best_alpha = float(a_best)
                best_epoch = int(epoch)
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_va_pred = va_pred.copy()
                if best_dbg is not None:
                    best_mean_term = float(best_dbg.get("mean_term", 0.0))
                    best_wcos = float(best_dbg.get("wcos", 0.0))
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    return best_score, best_alpha, best_epoch, best_state, best_va_pred, best_mean_term, best_wcos


# ================================
# GROUPED CV SPLIT
# - hold out entire perturbation identities
# ================================
N = Y.shape[0]

# Put the aligned per-row perturbation names here
PERT_NAMES = np.asarray(train_genes)

if len(PERT_NAMES) != N:
    raise ValueError(f"PERT_NAMES length {len(PERT_NAMES)} does not match N={N}")

groups = np.asarray([str(x).strip().upper() for x in PERT_NAMES])
unique_groups = np.unique(groups)

if len(unique_groups) < 2:
    raise ValueError(f"Need at least 2 unique perturbations, got {len(unique_groups)}")

N_SPLITS = min(8, len(unique_groups))
gkf = GroupKFold(n_splits=N_SPLITS)

print(f"[split] total rows={N} | unique perts={len(unique_groups)} | n_splits={N_SPLITS}")

oof_pred = np.zeros_like(Y, dtype=np.float32)
oof_hit  = np.zeros((N,), dtype=np.int32)

fold_scores = []
fold_scores_seeds = []
fold_epochs = []
fold_alphas = []

fold_mean_terms = []
fold_wcos = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(np.arange(N), groups=groups), 1):
    tr_groups = set(groups[tr_idx])
    va_groups = set(groups[va_idx])

    overlap = tr_groups & va_groups
    if len(overlap) != 0:
        raise ValueError(f"Leakage: train/val perturbation overlap found: {sorted(list(overlap))[:10]}")

    seed_scores = []
    seed_alphas = []
    seed_epochs = []
    seed_va_preds = []
    seed_mean_terms = []
    seed_wcos = []

    for s in MODEL_SEEDS:
        best_score, best_alpha, best_epoch, best_state, best_va_pred, best_mean_term, best_wcos_ = train_one_fold(
            tr_idx, va_idx, seed=int(s)
        )

        seed_scores.append(float(best_score))
        seed_alphas.append(float(best_alpha))
        seed_epochs.append(int(best_epoch))
        seed_va_preds.append(best_va_pred.astype(np.float32, copy=False))
        seed_mean_terms.append(float(best_mean_term))
        seed_wcos.append(float(best_wcos_))

    va_pred_mean = np.mean(np.stack(seed_va_preds, axis=0), axis=0).astype(np.float32)
    oof_pred[va_idx] = va_pred_mean
    oof_hit[va_idx] += 1

    fold_factor = float(N / len(va_idx))
    seed_scores_scaled = [sc * fold_factor for sc in seed_scores]
    fold_score_scaled = float(np.mean(seed_scores_scaled))

    fold_scores.append(fold_score_scaled)
    fold_scores_seeds.append(seed_scores_scaled)
    fold_alphas.append(float(np.mean(seed_alphas)))
    fold_epochs.append(int(np.median(seed_epochs)))
    fold_mean_terms.append(float(np.mean(seed_mean_terms)))
    fold_wcos.append(float(np.mean(seed_wcos)))

    seeds_str = ", ".join([f"{sc:.6f}" for sc in seed_scores_scaled])

    print(
        f"fold {fold}: score_mean={fold_score_scaled:.6f} "
        f"scores=[{seeds_str}] "
        f"alpha_mean={np.mean(seed_alphas):.3f} "
        f"epoch_median={int(np.median(seed_epochs))} "
        f"mean_term_mean={np.mean(seed_mean_terms):.5f} "
        f"wcos_mean={np.mean(seed_wcos):.5f} "
        f"val_n={len(va_idx)} "
        f"val_perts={len(va_groups)}"
    )

if not np.all(oof_hit == 1):
    print("[warn] OOF coverage not 1 everywhere. min/max:", int(oof_hit.min()), int(oof_hit.max()))

print("group-cv mean:", float(np.mean(fold_scores)), "std:", float(np.std(fold_scores)))
print("median best_epoch =", int(np.median(fold_epochs)))
print("mean_term overall:", float(np.mean(fold_mean_terms)), "wcos overall:", float(np.mean(fold_wcos)))

best_global_alpha = 0.0
best_global_score = -1e18
all_idx = np.arange(N, dtype=np.int64)

for a in ALPHA_GRID:
    pred_a = apply_shrink(oof_pred, delta_baseline, float(a))
    sc = score_delta(Y, pred_a, all_idx)["score"]
    if sc > best_global_score:
        best_global_score = sc
        best_global_alpha = float(a)

print("OOF global alpha:", best_global_alpha, "OOF score:", best_global_score)

ALPHA_SHRINK = best_global_alpha
MEDIAN_EPOCHS = int(np.median(fold_epochs))

[split] total rows=80 | unique perts=80 | n_splits=8
fold 1: score_mean=6.180133 scores=[6.300720, 6.171600, 6.068080] alpha_mean=0.720 epoch_median=45 mean_term_mean=0.22673 wcos_mean=0.34070 val_n=10 val_perts=10
fold 2: score_mean=6.052160 scores=[6.038400, 6.015840, 6.102240] alpha_mean=0.564 epoch_median=30 mean_term_mean=0.22131 wcos_mean=0.34202 val_n=10 val_perts=10
fold 3: score_mean=6.627227 scores=[7.031120, 6.512080, 6.338480] alpha_mean=0.712 epoch_median=50 mean_term_mean=0.23733 wcos_mean=0.34892 val_n=10 val_perts=10
fold 4: score_mean=5.453547 scores=[5.393680, 5.613360, 5.353600] alpha_mean=0.823 epoch_median=50 mean_term_mean=0.22208 wcos_mean=0.30693 val_n=10 val_perts=10
fold 5: score_mean=5.034213 scores=[4.979520, 5.095360, 5.027760] alpha_mean=0.796 epoch_median=20 mean_term_mean=0.22388 wcos_mean=0.28132 val_n=10 val_perts=10
fold 6: score_mean=7.797947 scores=[7.952160, 7.480240, 7.961440] alpha_mean=0.734 epoch_median=30 mean_term_mean=0.25015 wcos_mean=0.389

PROPER SPLIT
[split] total rows=80 | unique perts=80 | n_splits=8
fold 1: score_mean=6.180133 scores=[6.300720, 6.171600, 6.068080] alpha_mean=0.720 epoch_median=45 mean_term_mean=0.22673 wcos_mean=0.34070 val_n=10 val_perts=10
fold 2: score_mean=6.052160 scores=[6.038400, 6.015840, 6.102240] alpha_mean=0.564 epoch_median=30 mean_term_mean=0.22131 wcos_mean=0.34202 val_n=10 val_perts=10
fold 3: score_mean=6.627227 scores=[7.031120, 6.512080, 6.338480] alpha_mean=0.712 epoch_median=50 mean_term_mean=0.23733 wcos_mean=0.34892 val_n=10 val_perts=10
fold 4: score_mean=5.453547 scores=[5.393680, 5.613360, 5.353600] alpha_mean=0.823 epoch_median=50 mean_term_mean=0.22208 wcos_mean=0.30693 val_n=10 val_perts=10
fold 5: score_mean=5.034213 scores=[4.979520, 5.095360, 5.027760] alpha_mean=0.796 epoch_median=20 mean_term_mean=0.22388 wcos_mean=0.28132 val_n=10 val_perts=10
fold 6: score_mean=7.797947 scores=[7.952160, 7.480240, 7.961440] alpha_mean=0.734 epoch_median=30 mean_term_mean=0.25015 wcos_mean=0.38960 val_n=10 val_perts=10
fold 7: score_mean=6.719760 scores=[6.591680, 6.838240, 6.729360] alpha_mean=0.666 epoch_median=45 mean_term_mean=0.20949 wcos_mean=0.40094 val_n=10 val_perts=10
fold 8: score_mean=4.841040 scores=[5.032080, 4.830160, 4.660880] alpha_mean=0.761 epoch_median=20 mean_term_mean=0.19612 wcos_mean=0.30879 val_n=10 val_perts=10
group-cv mean: 6.088253333333334 std: 0.9145683991916626
median best_epoch = 37
mean_term overall: 0.22338606902639832 wcos overall: 0.3399041012845091
OOF global alpha: 0.7121211886405945 OOF score: 5.87839

fold 1: score_mean=7.498880 scores=[7.425360, 7.414080, 7.657200] alpha_mean=0.836 epoch_median=40 mean_term_mean=0.25446 wcos_mean=0.36837
fold 2: score_mean=4.873067 scores=[4.601920, 5.142080, 4.875200] alpha_mean=0.766 epoch_median=30 mean_term_mean=0.19078 wcos_mean=0.31913
fold 3: score_mean=5.353093 scores=[5.272960, 5.303200, 5.483120] alpha_mean=0.656 epoch_median=45 mean_term_mean=0.22702 wcos_mean=0.29473
fold 4: score_mean=5.151093 scores=[4.984960, 5.306560, 5.161760] alpha_mean=0.825 epoch_median=35 mean_term_mean=0.21110 wcos_mean=0.30504
fold 5: score_mean=5.696960 scores=[5.763440, 5.662720, 5.664720] alpha_mean=0.758 epoch_median=40 mean_term_mean=0.21538 wcos_mean=0.33069
fold 6: score_mean=7.648480 scores=[7.720720, 7.656960, 7.567760] alpha_mean=0.666 epoch_median=35 mean_term_mean=0.25774 wcos_mean=0.37094
fold 7: score_mean=7.205600 scores=[7.366560, 6.956560, 7.293680] alpha_mean=0.618 epoch_median=60 mean_term_mean=0.26947 wcos_mean=0.33420
fold 8: score_mean=5.130533 scores=[5.143120, 5.103840, 5.144640] alpha_mean=0.658 epoch_median=20 mean_term_mean=0.18872 wcos_mean=0.34006
cv mean: 6.069713333333333 std: 1.0974763301522472
median best_epoch = 37
mean_term overall: 0.2268340689759238 wcos overall: 0.3328950367428561
OOF global alpha: 0.7040404081344604 OOF score: 5.88905

fold 1: score_mean=7.138133 scores=[7.168640, 7.046960, 7.198800] alpha_mean=0.828 epoch_median=20 mean_term_mean=0.24446 wcos_mean=0.36502
fold 2: score_mean=4.736773 scores=[4.493280, 5.000320, 4.716720] alpha_mean=0.750 epoch_median=10 mean_term_mean=0.18752 wcos_mean=0.31562
fold 3: score_mean=5.111680 scores=[4.963600, 5.057600, 5.313840] alpha_mean=0.637 epoch_median=15 mean_term_mean=0.21900 wcos_mean=0.29172
fold 4: score_mean=5.010347 scores=[4.860400, 5.176800, 4.993840] alpha_mean=0.793 epoch_median=20 mean_term_mean=0.20987 wcos_mean=0.29839
fold 5: score_mean=5.537840 scores=[5.532960, 5.349920, 5.730640] alpha_mean=0.669 epoch_median=5 mean_term_mean=0.20483 wcos_mean=0.33803
fold 6: score_mean=7.319973 scores=[7.387120, 7.460240, 7.112560] alpha_mean=0.639 epoch_median=15 mean_term_mean=0.24968 wcos_mean=0.36647
fold 7: score_mean=6.999840 scores=[7.219280, 6.830720, 6.949520] alpha_mean=0.604 epoch_median=15 mean_term_mean=0.26436 wcos_mean=0.33094
fold 8: score_mean=4.917520 scores=[4.959440, 4.897680, 4.895440] alpha_mean=0.642 epoch_median=15 mean_term_mean=0.18732 wcos_mean=0.32817
cv mean: 5.846513333333332 std: 1.036732512871302
median best_epoch = 15
mean_term overall: 0.22087737287998777 wcos overall: 0.3292950072183771
OOF global alpha: 0.6797980070114136 OOF score: 5.77313

fold 1: score_mean=7.017947 scores=[7.098800, 6.910880, 7.044160] alpha_mean=0.844 epoch_median=40 mean_term_mean=0.24138 wcos_mean=0.36359
fold 2: score_mean=4.610133 scores=[4.709360, 4.397760, 4.723280] alpha_mean=0.769 epoch_median=25 mean_term_mean=0.18464 wcos_mean=0.31223
fold 3: score_mean=5.100587 scores=[5.250560, 4.946480, 5.104720] alpha_mean=0.637 epoch_median=45 mean_term_mean=0.21524 wcos_mean=0.29619
fold 4: score_mean=4.832907 scores=[4.764560, 4.846560, 4.887600] alpha_mean=0.831 epoch_median=55 mean_term_mean=0.20437 wcos_mean=0.29560
fold 5: score_mean=5.325973 scores=[5.313920, 5.290880, 5.373120] alpha_mean=0.726 epoch_median=40 mean_term_mean=0.20605 wcos_mean=0.32310
fold 6: score_mean=7.507493 scores=[7.412960, 7.686000, 7.423520] alpha_mean=0.680 epoch_median=35 mean_term_mean=0.25448 wcos_mean=0.36876
fold 7: score_mean=7.507067 scores=[7.462720, 7.658240, 7.400240] alpha_mean=0.650 epoch_median=25 mean_term_mean=0.27151 wcos_mean=0.34561
fold 8: score_mean=4.926133 scores=[5.001440, 4.778240, 4.998720] alpha_mean=0.642 epoch_median=25 mean_term_mean=0.18561 wcos_mean=0.33178
cv mean: 5.85353 std: 1.178911358306656
median best_epoch = 37
mean_term overall: 0.22040890942993646 wcos overall: 0.32960829792497587
OOF global alpha: 0.6959595680236816 OOF score: 5.62978

In [177]:
# ================================
# FULL REFIT + SUBMISSION BUILD
# - refits LowRankBasisHyperNet on all training perts
# - uses safer h5ad augmentation application
# - builds submission using released pert_id -> pert mapping
# ================================

import numpy as np
import pandas as pd
import torch

PERT_IDS_ALL_CSV = "Data/pert_ids_all.csv"   # put your released pert roster path here


def fit_full_model(seed, epochs_fixed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = LowRankBasisHyperNet(
        d_pert=Zt.shape[1],
        basis_init=BASIS_INIT_t,
        hidden=HNET_HIDDEN,
        dropout=HNET_DROPOUT
    ).to(device)

    model.set_gene_bias_init(GENE_BIAS_INIT_t)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    # --- h5ad augmentation mapping fit on ALL usable perts (no CV now) ---
    aug_enabled = False
    slopes_t = None
    intercepts_t = None
    rng_aug = None

    if ("AUGMENT_H5AD" in globals()) and AUGMENT_H5AD:
        assert (boot_raw is not None) and (h5_mean_raw is not None) and (boot_ok is not None), \
            "AUGMENT_H5AD=True but boot_raw/h5_mean_raw/boot_ok not built. Run the bootstrap precompute cell."

        fit_idx = np.where(np.asarray(boot_ok) == 1)[0]
        if len(fit_idx) >= AUG_MIN_FIT_PERTS:
            A = h5_mean_raw[fit_idx].astype(np.float32)
            B = Y[fit_idx].astype(np.float32)

            Am = A.mean(axis=0)
            Bm = B.mean(axis=0)
            Av = ((A - Am[None, :]) ** 2).mean(axis=0) + 1e-6
            Cov = ((A - Am[None, :]) * (B - Bm[None, :])).mean(axis=0)

            slopes = (Cov / Av).astype(np.float32)
            slopes = np.clip(
                slopes,
                float(AUG_SLOPE_CLAMP_MIN),
                float(AUG_SLOPE_CLAMP_MAX)
            ).astype(np.float32)
            intercepts = (Bm - slopes * Am).astype(np.float32)

            slopes_t = torch.tensor(slopes[None, :], device=device, dtype=torch.float32)
            intercepts_t = torch.tensor(intercepts[None, :], device=device, dtype=torch.float32)

            rng_aug = np.random.RandomState(seed + 2027)
            aug_enabled = True
            print(
                f"[aug/full] enabled: fit_perts={len(fit_idx)} "
                f"slope_mean={float(slopes.mean()):.6f} "
                f"slope_std={float(slopes.std()):.6f}"
            )
        else:
            print(f"[aug/full] disabled: not enough fit perts with h5ad coverage ({len(fit_idx)})")

    all_idx = np.arange(N, dtype=np.int64)
    epochs_run = int(epochs_fixed) if epochs_fixed is not None else int(MEDIAN_EPOCHS)

    boot_ok_arr = np.asarray(boot_ok).reshape(-1) if boot_ok is not None else None

    for epoch in range(1, epochs_run + 1):
        model.train()
        perm = np.arange(N)
        np.random.shuffle(perm)

        for start in range(0, N, BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t))
            dt_b = Yt.index_select(0, b_t)

            # Safer augmentation:
            # only apply augmented targets to rows in the batch that actually have h5ad coverage
            if aug_enabled and (slopes_t is not None):
                covered_mask_np = (boot_ok_arr[b] == 1)

                if np.any(covered_mask_np):
                    kk = rng_aug.randint(0, BOOT_K, size=len(b))
                    dt_raw_np = boot_raw[b, kk, :]
                    dt_raw_t = torch.tensor(dt_raw_np, device=device, dtype=torch.float32)

                    dt_aug = dt_raw_t * slopes_t + intercepts_t

                    covered_mask_t = torch.tensor(
                        covered_mask_np[:, None],
                        device=device,
                        dtype=torch.bool
                    )

                    dt_mix = (1.0 - float(AUG_P)) * dt_b + float(AUG_P) * dt_aug
                    dt_b = torch.where(covered_mask_t, dt_mix, dt_b)

            bw_b = baseline_wmae_t.index_select(0, b_t)

            loss_main = weighted_l1_like_rowweighted(
                dt_b, pred, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )

            loss_basis_drift = LAMBDA_BASIS_DRIFT * torch.mean(model.basis_delta * model.basis_delta)
            loss_basis_gain  = LAMBDA_BASIS_GAIN  * torch.mean((model.basis_gain - 1.0) ** 2)

            loss = loss_main + loss_basis_drift + loss_basis_gain

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if (epoch % EVAL_EVERY == 0) or (epoch == int(epochs_run)):
            model.eval()
            with torch.no_grad():
                pred_np = model(Zt).detach().cpu().numpy().astype(np.float32)

            pred_np = apply_shrink(pred_np, delta_baseline, float(ALPHA_SHRINK))
            s = score_delta(Y, pred_np, all_idx)

            if isinstance(s, dict) and ("score" in s):
                print(
                    f"[seed {seed}] epoch={epoch:4d} "
                    f"train_score={s['score']:.6f} "
                    f"wcos={s.get('wcos', float('nan')):.6f} "
                    f"mean_term={s.get('mean_term', float('nan')):.6f} "
                    f"alpha={float(ALPHA_SHRINK):.3f}"
                )
            else:
                print(f"[seed {seed}] epoch={epoch:4d} train_score={float(s):.6f} alpha={float(ALPHA_SHRINK):.3f}")

    model.eval()
    return model


# ================================
# REFIT ENSEMBLE ON ALL TRAINING PERTS
# ================================
models = [fit_full_model(int(sd), epochs_fixed=int(MEDIAN_EPOCHS)) for sd in MODEL_SEEDS]
print("Refit models:", len(models))


def predict_delta_gene(gene_symbol: str) -> np.ndarray:
    z = torch.tensor(emb_pert(gene_symbol)[None, :].astype(np.float32), device=device)
    preds = []
    with torch.no_grad():
        for m in models:
            y = m(z).detach().cpu().numpy().astype(np.float32)[0]
            preds.append(y)

    yhat = np.mean(np.stack(preds, axis=0), axis=0).astype(np.float32)
    yhat = apply_shrink(yhat[None, :], delta_baseline, float(ALPHA_SHRINK))[0].astype(np.float32)
    return yhat


# ================================
# LOAD RELEASED PERT ROSTER
# - uses pert_id -> pert, not old val_map
# ================================
pert_all = pd.read_csv(PERT_IDS_ALL_CSV).copy()
pert_all.columns = [str(c).strip().lower() for c in pert_all.columns]

required_cols = ["pert_id", "pert"]
for c in required_cols:
    if c not in pert_all.columns:
        raise ValueError(f"Missing required column '{c}' in {PERT_IDS_ALL_CSV}")

pert_all["pert_id"] = pert_all["pert_id"].astype(str)
pert_all["pert"] = pert_all["pert"].astype(str).str.strip().str.upper()

released_pid_to_gene = dict(zip(pert_all["pert_id"], pert_all["pert"]))

print(f"[released] rows={len(pert_all)} unique_pert_ids={pert_all['pert_id'].nunique()} unique_perts={pert_all['pert'].nunique()}")


# ================================
# BUILD SUBMISSION FROM df_sub
# ================================
sub = df_sub.copy()
sub["pert_id"] = sub["pert_id"].astype(str)

sub_gene_cols = [c for c in sub.columns if c != "pert_id"]

idx_map = {str(g).upper(): i for i, g in enumerate(gene_columns)}
perm = [idx_map[str(g).upper()] for g in sub_gene_cols]

# default everything to baseline first
sub.loc[:, sub_gene_cols] = np.tile(delta_baseline[perm][None, :], (len(sub), 1))

hit = 0
miss = []

for pid in sub["pert_id"].tolist():
    gene = released_pid_to_gene.get(str(pid), None)
    if gene is None:
        miss.append(str(pid))
        continue

    vec = predict_delta_gene(gene)[perm]
    m = (sub["pert_id"] == str(pid))
    if m.any():
        sub.loc[m, sub_gene_cols] = vec[None, :]
        hit += int(m.sum())

out_path = "basis_hypernet_submission2.csv"
sub.to_csv(out_path, index=False)

print("[ok] wrote:", out_path)
print("[sub] filled rows:", hit, "/", len(sub))
print("[sub] missing pert_ids from released map:", len(miss))
if len(miss):
    print(miss[:20])

[aug/full] enabled: fit_perts=80 slope_mean=0.980505 slope_std=0.062581
[seed 90] epoch=   5 train_score=2.708160 wcos=0.407938 mean_term=0.082983 alpha=0.712
[seed 90] epoch=  10 train_score=6.095980 wcos=0.377288 mean_term=0.201967 alpha=0.712
[seed 90] epoch=  15 train_score=7.683490 wcos=0.349760 mean_term=0.274599 alpha=0.712
[seed 90] epoch=  20 train_score=8.130000 wcos=0.357913 mean_term=0.283938 alpha=0.712
[seed 90] epoch=  25 train_score=8.377430 wcos=0.352849 mean_term=0.296778 alpha=0.712
[seed 90] epoch=  30 train_score=8.527880 wcos=0.354717 mean_term=0.300517 alpha=0.712
[seed 90] epoch=  35 train_score=8.693090 wcos=0.355731 mean_term=0.305466 alpha=0.712
[seed 90] epoch=  37 train_score=8.767530 wcos=0.354943 mean_term=0.308766 alpha=0.712
[aug/full] enabled: fit_perts=80 slope_mean=0.980505 slope_std=0.062581
[seed 70] epoch=   5 train_score=2.584520 wcos=0.414350 mean_term=0.077969 alpha=0.712
[seed 70] epoch=  10 train_score=5.928790 wcos=0.381951 mean_term=0.19403

[aug/full] enabled: fit_perts=80 slope_mean=0.980505 slope_std=0.062581
[seed 90] epoch=   5 train_score=2.681240 wcos=0.408322 mean_term=0.082081 alpha=0.704
[seed 90] epoch=  10 train_score=6.061870 wcos=0.378096 mean_term=0.200408 alpha=0.704
[seed 90] epoch=  15 train_score=7.685420 wcos=0.350815 mean_term=0.273842 alpha=0.704
[seed 90] epoch=  20 train_score=8.107920 wcos=0.359019 mean_term=0.282294 alpha=0.704
[seed 90] epoch=  25 train_score=8.359450 wcos=0.353999 mean_term=0.295180 alpha=0.704
[seed 90] epoch=  30 train_score=8.506110 wcos=0.355844 mean_term=0.298800 alpha=0.704
[seed 90] epoch=  35 train_score=8.669020 wcos=0.356850 mean_term=0.303665 alpha=0.704
[seed 90] epoch=  37 train_score=8.744260 wcos=0.356063 mean_term=0.306978 alpha=0.704
[aug/full] enabled: fit_perts=80 slope_mean=0.980505 slope_std=0.062581
[seed 70] epoch=   5 train_score=2.558600 wcos=0.414686 mean_term=0.077125 alpha=0.704
[seed 70] epoch=  10 train_score=5.893850 wcos=0.382737 mean_term=0.192490 alpha=0.704
[seed 70] epoch=  15 train_score=7.628600 wcos=0.353522 mean_term=0.269735 alpha=0.704
[seed 70] epoch=  20 train_score=8.098990 wcos=0.358520 mean_term=0.282376 alpha=0.704
[seed 70] epoch=  25 train_score=8.285980 wcos=0.356162 mean_term=0.290808 alpha=0.704
[seed 70] epoch=  30 train_score=8.429830 wcos=0.354811 mean_term=0.296983 alpha=0.704
[seed 70] epoch=  35 train_score=8.586060 wcos=0.357262 mean_term=0.300412 alpha=0.704
[seed 70] epoch=  37 train_score=8.671700 wcos=0.357490 mean_term=0.303215 alpha=0.704
[aug/full] enabled: fit_perts=80 slope_mean=0.980505 slope_std=0.062581
[seed 80] epoch=   5 train_score=2.676080 wcos=0.410122 mean_term=0.081564 alpha=0.704
[seed 80] epoch=  10 train_score=6.012220 wcos=0.377446 mean_term=0.199109 alpha=0.704
[seed 80] epoch=  15 train_score=7.583520 wcos=0.350164 mean_term=0.270713 alpha=0.704
[seed 80] epoch=  20 train_score=8.073780 wcos=0.358641 mean_term=0.281402 alpha=0.704
[seed 80] epoch=  25 train_score=8.346960 wcos=0.354339 mean_term=0.294455 alpha=0.704
[seed 80] epoch=  30 train_score=8.480840 wcos=0.355786 mean_term=0.297962 alpha=0.704
[seed 80] epoch=  35 train_score=8.657140 wcos=0.355530 mean_term=0.304375 alpha=0.704
[seed 80] epoch=  37 train_score=8.714640 wcos=0.356772 mean_term=0.305329 alpha=0.704
Refit models: 3
[ok] wrote: basis_hypernet_submission.csv | filled: 60